# Sampling phone occurrences from LibriSpeech

_[Marianne de Heer Kloots](https://mdhk.net/), Apr 2026_

This notebook walks through sampling a small set of recordings from the clean validation (`dev.clean`) subset of LibriSpeech, to be used for probing phone information from audio encoder models. We will choose a set of phones and speakers such that we can sample an equal number of phone occurrences for each speaker, and save our analysis sample to disk as a dataframe with time-aligned phone annotations (`data/libri_phone_sample/libri_phone_sample.csv`) and accompanying audio files (one `.wav` file for each sampled recording).

To be able to do this, we first need a set of time-aligned phonetic annotations for the LibriSpeech corpus (i.e. annotations that tell us where each phone pronunciation starts and ends in the audio recording). Such annotations can be generated automatically from any set of paired audio recordings and text transcriptions, using _forced alignment_ tools (e.g. [MFA](https://montreal-forced-aligner.readthedocs.io/en/latest/), [WebMAUS](https://clarin.phonetik.uni-muenchen.de/BASWebServices/interface/WebMAUSBasic)). Here, we'll make use of an [existing set of alignments for the LibriSpeech corpus](https://zenodo.org/records/2619474) published by [Lugosch et al. (2019)](https://www.isca-archive.org/interspeech_2019/lugosch19_interspeech.html); for convenience I've uploaded a slightly reformatted and extended version of these as a dataset on the HuggingFace hub ([mariannedhk/librispeech_phones](https://huggingface.co/datasets/mariannedhk/librispeech_phones)). We'll select our sample of phones based on this dataset of aligned phone annotations, and then extract all audio recordings containing the sampled phones from the [openslr/librispeech_asr](https://huggingface.co/datasets/openslr/librispeech_asr) dataset.

## Installs & set-up

In [ ]:
!pip install "librosa>=0.11.0" "pandas>=2.2.2" "datasets>=4.0.0" "torchcodec==0.10.0"

In [1]:
from datasets import load_dataset
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import shutil
import IPython.display as ipd
from IPython.display import Audio
from tqdm import tqdm
from pathlib import Path

In [2]:
# set a seed for reproducability
SEED = 12

# choose a directory to save data to (default is the current directory)
SAVEDIR = Path('')

## Generate phone sample
Here we'll sample a set of phone occurrences from the LibriSpeech subset that fits our requirements: we want to sample 35 different phone classes for which at least 10 occurrences per phone are available for each of 10 speakers.

We'll first load the `librispeech_phones` dataset and register some information on it (the number of phones per speaker) such that we can generate the subset.

In [3]:
# load phone alignments for the LibriSpeech dev.clean set 
# (convert to pandas for easier data handling)
libri_dev_clean_phones = pd.DataFrame(
    load_dataset("mariannedhk/librispeech_phones", "dev.clean", cache_dir=SAVEDIR/'data/hf')['dev.clean']
)

Generating dev.clean split:   0%|          | 0/193644 [00:00<?, ? examples/s]

In [4]:
def libri_phone_speaker_count(speakers, phones, phone_alignments=libri_dev_clean_phones):
    """
    Count the number of occurrences per speaker per phone (the number of times each speaker pronounces each phone).
    """
    phone_counts = {p: {} for p in phones}
    for speaker_id in tqdm(speakers.keys(), desc='counting phones per speaker'):
        speaker_sex = speakers[speaker_id]
        speaker_rows = phone_alignments[phone_alignments['speaker_id'] == speaker_id]
        speaker_phone_count = pd.Series(speaker_rows['phone_ipa']).value_counts()
        for phone in phones:
            if phone in speaker_phone_count:
                phone_counts[phone][f'{speaker_id}_{speaker_sex}'] = speaker_phone_count[phone]
            else:
                phone_counts[phone][f'{speaker_id}_{speaker_sex}'] = 0
    count_df = pd.DataFrame.from_dict(phone_counts, orient='index')
    return count_df

In [5]:
phones = libri_dev_clean_phones['phone_ipa'].unique()
speakers = {sp_id: sp_s for sp_id, sp_s in list(zip(libri_dev_clean_phones['speaker_id'], libri_dev_clean_phones['speaker_sex']))}
phone_counts = libri_phone_speaker_count(speakers, phones)

counting phones per speaker: 100%|██████████| 40/40 [00:00<00:00, 484.39it/s]


Below we select a set of phones and speakers that fits our requirements (by specifying `Nphones = 35` for the number of phone classes, `Noccurrences = 10` for the number of occurrences per phone per speaker, and `Nspeakers = 10` for the number of speakers to include). 

To balance speaker sex, we randomly sample half of the included speakers to be male and the other half to be female.

In [6]:
Nphones = 35
Noccurrences = 10
Nspeakers = 10

# include the N most frequent phones
selected_phones = pd.Series(libri_dev_clean_phones['phone_ipa']).value_counts().index[:Nphones]
phone_counts = phone_counts.loc[selected_phones]

# only include speakers who have enough occurrences of each phone
phone_counts.loc['Nphones_above_threshold'] = [
    sum([True if phone_counts.loc[p, col] >= Noccurrences else False for p in phone_counts.index])
    for col in phone_counts.columns
]
possible_speakers = {
    'F': [int(c.split('_')[0]) for c in phone_counts.columns 
          if (phone_counts.loc['Nphones_above_threshold', c] >= Nphones) 
          and c.endswith('F')],
    'M': [int(c.split('_')[0]) for c in phone_counts.columns 
          if (phone_counts.loc['Nphones_above_threshold', c] >= Nphones) 
          and c.endswith('M')]
}

# randomly sample 5 female and 5 male speakers from the speakers with enough occurrences of each phone
np.random.seed(SEED)
selected_speakers = np.random.choice(possible_speakers['F'], Nspeakers//2, replace=False).tolist() +\
                    np.random.choice(possible_speakers['M'], Nspeakers//2, replace=False).tolist()

Now that we've defined a speaker and phone selection, we can generate the sample! The sample will include `Nphones * Noccurrences * Nspeakers` phone occurrences in total, i.e. 35\*10\*10 = 3500 for the requirements we specified here.

In [7]:
def libri_phone_sample(speaker_selection, phone_selection, Noccurrences, 
                       phone_alignments=libri_dev_clean_phones):
    """
    Sample a set of phone occurrences based on the LibriSpeech phone alignments,
    including only speakers in speaker_selection and phones in phone_selection,
    sampling Noccurrences per phone per speaker.
    """
    speaker_samples = []
    for speaker_id in tqdm(speaker_selection, desc='sampling phones per speaker'):
        speaker_phone_rows = phone_alignments[
            (phone_alignments['speaker_id'] == speaker_id) &
            (phone_alignments['phone_ipa'].isin(phone_selection))
        ]
        speaker_samples.append(speaker_phone_rows.groupby('phone_ipa').sample(Noccurrences))
    phone_sample = pd.concat(speaker_samples).reset_index(drop=True)
    return phone_sample

Annotations for the generated phone sample are stored as a DataFrame (which is effectively a subset of rows from the larger `librispeech_phones` dataset). This DataFrame includes columns specifying what audio file each sampled phone occurs in (`file_id`) and the phone's start and end time within that file (`start_time`, `end_time`), which is what we'll need to extract model activations corresponding to specific phone segments.

In [8]:
phone_sample = libri_phone_sample(selected_speakers, selected_phones, Noccurrences)
phone_sample

sampling phones per speaker: 100%|██████████| 10/10 [00:00<00:00, 86.37it/s]


,phone,phone_stress,phone_ipa,phone_position,phone_broad_category,phone_fine_category,start_time,end_time,speaker_id,speaker_sex,file_id,subset
0,AY,AY1,ai,0,vowel,diphthong,6.01,6.23,3853,F,3853-163249-0016,dev.clean
1,AY,AY1,ai,1,vowel,diphthong,2.27,2.46,3853,F,3853-163249-0001,dev.clean
2,AY,AY1,ai,1,vowel,diphthong,13.30,13.48,3853,F,3853-163249-0012,dev.clean
3,AY,AY2,ai,1,vowel,diphthong,2.45,2.58,3853,F,3853-163249-0046,dev.clean
4,AY,AY1,ai,1,vowel,diphthong,5.33,5.50,3853,F,3853-163249-0019,dev.clean
...,...,...,...,...,...,...,...,...,...,...,...,...
3495,TH,TH,θ,0,consonant,fricative,10.56,10.68,6241,M,6241-61943-0027,dev.clean
3496,TH,TH,θ,2,consonant,fricative,7.57,7.64,6241,M,6241-66616-0011,dev.clean
3497,TH,TH,θ,2,consonant,fricative,5.48,5.57,6241,M,6241-66616-0010,dev.clean
3498,TH,TH,θ,3,consonant,fricative,0.82,0.86,6241,M,6241-66616-0024,dev.clean


## Find audio for sampled phones
In order to extract activations from audio encoder models, we'll obviously also need the actual audio recordings. 

Here we extract all needed audio files, specified by the `file_id` column of our phone sample, from the LibriSpeech dev.clean dataset.

In [9]:
# load the LibriSpeech dataset of audiobook recordings
# (use streaming mode to avoid downloading the full dataset; we'll only download the files we need)
libri_dev_clean = load_dataset("openslr/librispeech_asr", split="validation.clean", cache_dir=SAVEDIR/'data/hf', streaming=True)

def libri_audios(phone_sample, libri_data=libri_dev_clean):
    """
    Retrieve the audio data for all LibriSpeech recordings in phone_sample.
    """
    file_ids = list(phone_sample['file_id'].unique())
    audios = {fid: {'array': None, 'sampling_rate': None} for fid in file_ids}
    for item in tqdm(libri_data, 'extracting audios from LibriSpeech'):
        if item['id'] in file_ids:
            audios[item['id']]['array'] = item['audio']['array']
            audios[item['id']]['sampling_rate'] = item['audio']['sampling_rate']
    return audios

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

In [10]:
audios = libri_audios(phone_sample)

extracting audios from LibriSpeech: 2703it [00:07, 369.64it/s]


We can listen to the recordings for some sampled phone occurrences to see if everything went according to plan:

In [11]:
idx = 1982

row = phone_sample.loc[idx]
audio = audios[row['file_id']]['array']
samp_freq = audios[row['file_id']]['sampling_rate']
phone_start_time = row['start_time']
phone_end_time = row['end_time']
phone_start_sample = int(np.floor(phone_start_time * samp_freq))
phone_end_sample = int(np.ceil(phone_end_time * samp_freq))

print(f'full recording (phone [{row["phone_ipa"]}] at {phone_start_time}-{phone_end_time} s):')
ipd.display(Audio(audio, rate=samp_freq))
print(f'isolated phone (ipa [{row["phone_ipa"]}], arpabet [{row["phone"]}]):')
ipd.display(Audio(audio[phone_start_sample:phone_end_sample], rate=samp_freq))

full recording (phone [æ] at 0.49-0.68 s):


isolated phone (ipa [æ], arpabet [AE]):


## Save to disk

We'll now save the sampled set of recordings and phone annotations to disk, for later use.

In [12]:
def audios_to_disk(audios, save_dir, remove_existing=True, samp_freq=16000):
    """
    Write the audio data to disk as wav files.
    """
    save_dir = Path(save_dir)
    if save_dir.exists():
        if remove_existing:
            print(f'Target directory ({save_dir}) exists; removing existing contents before writing new data')
            shutil.rmtree(save_dir)
        else:
            print(f'Target directory ({save_dir}) exists; replacing/adding to existing contents')
    save_dir.mkdir(parents=True, exist_ok=True)
    print(f'Saving {len(audios)} audio files...')
    for audio_name in audios.keys():
        y = audios[audio_name]['array']
        sr = audios[audio_name]['sampling_rate']
        if sr == samp_freq:
            sf.write(save_dir / f'{audio_name}.wav', y, sr)
        else:
            print(f'\tSampling rate for {audio_name} does not match target sampling rate ({samp_freq} Hz); resampling to target')
            y = librosa.resample(y, orig_sr=sr, target_sr=samp_freq)
            sf.write(save_dir / f'{audio_name}.wav', y, samp_freq)
    print(f'Done! All files saved to {save_dir}')

In [13]:
audios_to_disk(audios, SAVEDIR/'data/libri_phone_sample/audio')

Saving 638 audio files...
Done! All files saved to data/libri_phone_sample/audio


In [14]:
phone_sample.to_csv(SAVEDIR/'data/libri_phone_sample/libri_phone_sample.csv', index=False)